In [1]:
import pandas as pd

holiday_events = pd.read_csv('data/holidays_events.csv', parse_dates=['date'])
oil = pd.read_csv('data/oil.csv', parse_dates=['date'])
stores = pd.read_csv('data/stores.csv')
transactions = pd.read_csv('data/transactions.csv', parse_dates=['date'])
train_data = pd.read_csv('data/train.csv', parse_dates=['date'])
test_data = pd.read_csv('data/test.csv', parse_dates=['date'])

In [2]:
oil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[ns]
 1   dcoilwtico  1175 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 19.2 KB


In [3]:
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()

In [4]:
oil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[ns]
 1   dcoilwtico  1218 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 19.2 KB


In [5]:
train_data = pd.merge(train_data, stores, on='store_nbr', how='left')
test_data = pd.merge(test_data, stores, on='store_nbr', how='left')

In [6]:
train_data = pd.merge(train_data, oil, on='date', how='left')
test_data = pd.merge(test_data, oil, on='date', how='left')

In [7]:
train_data

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,dcoilwtico
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0,Quito,Pichincha,D,13,93.14
1,1,2013-01-01,1,BABY CARE,0.000,0,Quito,Pichincha,D,13,93.14
2,2,2013-01-01,1,BEAUTY,0.000,0,Quito,Pichincha,D,13,93.14
3,3,2013-01-01,1,BEVERAGES,0.000,0,Quito,Pichincha,D,13,93.14
4,4,2013-01-01,1,BOOKS,0.000,0,Quito,Pichincha,D,13,93.14
...,...,...,...,...,...,...,...,...,...,...,...
3000883,3000883,2017-08-15,9,POULTRY,438.133,0,Quito,Pichincha,B,6,47.57
3000884,3000884,2017-08-15,9,PREPARED FOODS,154.553,1,Quito,Pichincha,B,6,47.57
3000885,3000885,2017-08-15,9,PRODUCE,2419.729,148,Quito,Pichincha,B,6,47.57
3000886,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8,Quito,Pichincha,B,6,47.57


In [8]:
train_data_idx = len(train_data)

In [9]:
df = pd.concat([train_data, test_data], axis=0, ignore_index=True)
df = df.sort_values(['date', 'store_nbr', 'family']).reset_index(drop=True)

In [10]:
df = pd.merge(df, transactions, on=['date', 'store_nbr'], how='left')

In [11]:
df['transactions'] = df['transactions'].fillna(0)

In [12]:
df['transactions_lag_16'] = df.groupby('store_nbr')['transactions'].transform(lambda x: x.shift(16))
df['transactions_lag_30'] = df.groupby('store_nbr')['transactions'].transform(lambda x: x.shift(30))

In [13]:
df['transactions_roll_mean_7'] = df.groupby('store_nbr')['transactions_lag_16'].transform(lambda x: x.rolling(7).mean())
df['transactions_roll_std_7'] = df.groupby('store_nbr')['transactions_lag_16'].transform(lambda x: x.rolling(7).std())

In [14]:
df['transactions_lag_16'] = df['transactions_lag_16'].fillna(df['transactions_lag_16'].mean())
df['transactions_lag_30'] = df['transactions_lag_30'].fillna(df['transactions_lag_30'].mean())
df['transactions_roll_mean_7'] = df['transactions_roll_mean_7'].fillna(df['transactions_roll_mean_7'].mean())
df['transactions_roll_std_7'] = df['transactions_roll_std_7'].fillna(df['transactions_roll_std_7'].mean())

df['dcoilwtico'] = df['dcoilwtico'].fillna(df['dcoilwtico'].mean())

In [15]:
df['dayofweek'] = df['date'].dt.dayofweek
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['is_salary_day'] = df['day'].isin([15, 16, 30, 31]).astype(int)

In [16]:
df = df.drop(columns=['transactions'])

train_final = df.iloc[:train_data_idx].copy()
test_final = df.iloc[train_data_idx:].copy()

In [17]:
train_final

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,dcoilwtico,transactions_lag_16,transactions_lag_30,transactions_roll_mean_7,transactions_roll_std_7,dayofweek,day,month,is_salary_day
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
1,1,2013-01-01,1,BABY CARE,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
2,2,2013-01-01,1,BEAUTY,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
3,3,2013-01-01,1,BEVERAGES,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
4,4,2013-01-01,1,BOOKS,0.000,0,Quito,Pichincha,D,13,93.14,1541.604652,1541.989572,1541.768831,18.043303,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3000883,3000751,2017-08-15,54,POULTRY,59.619,0,El Carmen,Manabi,C,3,47.57,802.000000,818.000000,802.000000,0.000000,1,15,8,1
3000884,3000752,2017-08-15,54,PREPARED FOODS,94.000,0,El Carmen,Manabi,C,3,47.57,802.000000,818.000000,802.000000,0.000000,1,15,8,1
3000885,3000753,2017-08-15,54,PRODUCE,915.371,76,El Carmen,Manabi,C,3,47.57,802.000000,802.000000,802.000000,0.000000,1,15,8,1
3000886,3000754,2017-08-15,54,SCHOOL AND OFFICE SUPPLIES,0.000,0,El Carmen,Manabi,C,3,47.57,802.000000,802.000000,802.000000,0.000000,1,15,8,1


In [18]:
import numpy as np

train_final['target_log'] = np.log1p(train_final['sales'])

In [19]:
train_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 20 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   id                        int64         
 1   date                      datetime64[ns]
 2   store_nbr                 int64         
 3   family                    object        
 4   sales                     float64       
 5   onpromotion               int64         
 6   city                      object        
 7   state                     object        
 8   type                      object        
 9   cluster                   int64         
 10  dcoilwtico                float64       
 11  transactions_lag_16       float64       
 12  transactions_lag_30       float64       
 13  transactions_roll_mean_7  float64       
 14  transactions_roll_std_7   float64       
 15  dayofweek                 int32         
 16  day                       int32         
 17  month   

In [20]:
start_date = '2016-01-01'
train_final = train_final[train_final['date'] >= start_date]

In [21]:
split_date = '2017-07-31'
train_data = train_final[train_final['date'] < split_date]
val_data = train_final[train_final['date'] >= split_date]

In [22]:
features = [
    'store_nbr', 'onpromotion', 'cluster', 'dcoilwtico',
    'transactions_lag_16', 'transactions_lag_30',
    'transactions_roll_mean_7', 'transactions_roll_std_7',
    'dayofweek', 'day', 'month', 'is_salary_day', 'family',
    'city', 'state', 'type'
]

X_train = train_data[features]
y_train = train_data['target_log']

X_val = val_data[features]
y_val = val_data['target_log']

In [23]:
cat_features = ['family', 'city', 'state', 'type']

In [24]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.001,
    depth=5,
    eval_metric='RMSE',
    early_stopping_rounds=100,
    random_seed=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    cat_features=cat_features,
    verbose=100
)

0:	learn: 2.5797911	test: 2.5271843	best: 2.5271843 (0)	total: 1.05s	remaining: 26m 11s
100:	learn: 2.3843136	test: 2.3304321	best: 2.3304321 (100)	total: 21.2s	remaining: 4m 53s
200:	learn: 2.2089482	test: 2.1542471	best: 2.1542471 (200)	total: 41s	remaining: 4m 25s
300:	learn: 2.0525077	test: 1.9969699	best: 1.9969699 (300)	total: 57.7s	remaining: 3m 49s
400:	learn: 1.9121500	test: 1.8564094	best: 1.8564094 (400)	total: 1m 15s	remaining: 3m 25s
500:	learn: 1.7863305	test: 1.7307215	best: 1.7307215 (500)	total: 1m 33s	remaining: 3m 6s
600:	learn: 1.6746604	test: 1.6189462	best: 1.6189462 (600)	total: 1m 50s	remaining: 2m 45s
700:	learn: 1.5759940	test: 1.5199224	best: 1.5199224 (700)	total: 2m 16s	remaining: 2m 36s
800:	learn: 1.4885146	test: 1.4323167	best: 1.4323167 (800)	total: 2m 48s	remaining: 2m 27s


KeyboardInterrupt: 

In [177]:
y_pred = model.predict(test_final[features])
y_pred = np.expm1(y_pred)
preds_real = np.clip(y_pred, 0, None)

In [179]:
submission = pd.DataFrame({
    'id': test_final['id'],
    'sales': preds_real
})

In [181]:
submission.to_csv('submission.csv', index=False)